In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from tqdm import trange

In [ ]:
class BaseModel(nn.Module):
    def forward(self, data):
        raise NotImplementedError("Forward method must be implemented.")

# Function to dynamically import model classes
def import_model_class(module_name, class_name):
    module = __import__(module_name, fromlist=[class_name])
    return getattr(module, class_name)

In [ ]:
model_classes = [
    ("ral_model", "RAL", {"input_dim": 64, "output_dim": 16}), 
    ("gcn_baseline_model", "GCNBaseline", {"input_dim": 64, "hidden_dim": 16,"output_dim": 1})
]

In [ ]:
models = []
for module_name, class_name, params in model_classes:
    model_class = import_model_class(module_name, class_name)
    model_instance = model_class(**params)
    models.append(model_instance)

# Load models or initialize if not present
for model in models:
    model_name = model.__class__.__name__.lower()
    if os.path.exists(f"{model_name}.pth"):
        model.load_state_dict(torch.load(f"{model_name}.pth"))

criterion = nn.MSELoss()

In [ ]:

# Training loop
if not all(os.path.exists(f"{model.__class__.__name__.lower()}.pth") for model in models):
    print("Training ...")
    with open("train.pkl", "rb") as f:
        graphs = pickle.load(f)
    train_loader = DataLoader(graphs, batch_size=32, shuffle=True)
    
    optimizers = [torch.optim.Adam(model.parameters(), lr=0.01) for model in models]
    num_epochs = 10
    loop = trange(num_epochs, desc="Training")
    
    for epoch in loop:
        losses = []
        for data in train_loader:
            for model, optimizer in zip(models, optimizers):
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, data.y)
                losses.append(loss.item())
                loss.backward()
                optimizer.step()
        
        loop.set_postfix({"MSE": np.mean(losses)})

    # Save models
    for model in models:
        torch.save(model.state_dict(), f"{model.__class__.__name__.lower()}.pth")

In [ ]:
def visualize_graph(G, original_labels, predicted_labels):
    # G = nx.DiGraph()
    # # Filter nodes and edges for the given graph_id
    # nodes = node_df[node_df['graph_id'] == graph_id]
    # edges = edge_df[edge_df['graph_id'] == graph_id]
    
    # # Add nodes and edges to the graph
    # G.add_nodes_from(nodes['node_id'].tolist())
    # for _, edge in edges.iterrows():
    #     G.add_edge(edge['source_node'], edge['target_node'], weight=edge['weight'])
    
    # Add layer attribute to nodes based on topological generations
    layers = {node: i for i, layer in enumerate(nx.topological_generations(G)) for node in layer}
    nx.set_node_attributes(G, layers, 'layer')

    labels = {
        node: f"y: {round(G.nodes[node]['y'], 2)}, value: {round(G.nodes[node]['value'], 2)}, diff: {round(original_labels[node] - predicted_labels[node], 2)}"
        for node in G.nodes
    } 

    # Draw the graph with multipartite layout
    pos = nx.multipartite_layout(G, subset_key='layer')  # Use multipartite layout for visualization
    plt.figure(figsize=(10, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=500, font_size=10, font_weight='bold', labels=labels)
    
    # Display edge weights
    labels = {edge: round(weight, 2) for edge, weight in nx.get_edge_attributes(G, 'weight').items()}  # Get edge weights and round to 2 decimal places
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)  # Draw edge labels
    plt.title(f"Graph Visualization")
    plt.show()

In [ ]:
with open("test.pkl", "rb") as f:
    graphs = pickle.load(f)

test_loader = DataLoader(graphs, batch_size=1000, shuffle=False)

for model in models:
    model.eval()
    with torch.no_grad():
        test_losses = []
        for data in test_loader:
            output = model(data)
            loss = criterion(output, data.y)
            test_losses.append(loss.item())
        print(f"{model.__class__.__name__} Test Loss: {np.mean(test_losses)}")


